# 5G Network Intelligence - Positive Quantile Resolution

## Hypothesis and prior evidence

Positive targets contribute over 92% of hurdle absolute error on development. Test whether the original sparse conditional grid loses useful resolution when interpolating the mixture median. More quantiles may add estimation noise or alter crossing repair; improvement is not assumed.

## Temporal protocol

Keep fit/stop/score boundaries 335/383/443 and 395/443/503. Every original observed beam target and per-horizon origin is scored. Stopping selects rounds only. Both windows are reused development evidence, not independent generalization tests. Validation 504-671, final holdout and production are excluded.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = '0a6335f270d90b0926c129835bfe7dba1bd57b96'


## Controlled intervention

Reuse the exact saved classifier and quantiles 0.05, 0.1, 0.25, 0.5. Add fixed levels 0.15, 0.2, 0.3, 0.35, 0.4, 0.45 with the original LightGBM parameters and patience. This is 36 new cloud fits; no grid selection, threshold change or base-component retraining. Sorted crossing repair is unchanged, so the intervention measures the complete finer-grid pipeline, not interpolation error in isolation. Full implementation: `scripts/train_hurdle_quantile_resolution.py`.


## Environment, source and canonical panel

Attach the completed original hurdle worker. Verify its artifact manifest before fitting. Readable trainer/config files come from a separate private MIT source dataset.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('hurdle_development_scores.csv') if (p.parent.parent/'artifacts/hurdle_research/manifest.json').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one completed hurdle output')
    SOURCE = roots[0]/'artifacts/hurdle_research'
    PROJECT = Path('/kaggle/working/quantile_resolution')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    patches=list(Path('/kaggle/input').rglob('train_hurdle_quantile_resolution.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one study source dataset')
    patch=patches[0].parent
    shutil.copyfile(patch/'train_hurdle_quantile_resolution.py','scripts/train_hurdle_quantile_resolution.py')
    shutil.copyfile(patch/'hurdle_quantile_resolution.yaml','configs/hurdle_quantile_resolution.yaml')
    Path('reports').mkdir(exist_ok=True)


## Training and debugging

Only positive targets enter each additional quantile fit. Stopping data is separate from score data. Save each new component immediately and preserve logs on failure. Original forecasts must replay exactly before comparing the refined grid. Checkpoints remain in Kaggle Output.


In [ ]:
if RUN_TRAINING:
    with Path('reports/hurdle_quantile_resolution_training.log').open('w') as log:
        process=subprocess.Popen([sys.executable,'-u','-m','scripts.train_hurdle_quantile_resolution','--source-root',str(SOURCE)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/quantile_resolution_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/hurdle_quantile_resolution'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError(f'Hosted fitting failed: {status}; inspect saved log')
else:
    print('Review mode: no fitting, calibration or holdout evaluation.')


## Evaluation and limitations

Report original-unit MAE and full-population WAPE (ratio, not percent). Keep negative results and all six fold/horizon comparisons. No automatic validation refit or promotion. One seed and reused windows cannot certify unseen-data accuracy or the WAPE <10% objective.


In [ ]:
paths=[Path('reports/hurdle_quantile_resolution_development.csv'),Path('../reports/hurdle_quantile_resolution_development.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('hurdle_quantile_resolution_development.csv'))
path=next((p for p in paths if p.is_file()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed study report attached.')


## Completed v1 result: finer grid rejected

Hosted Kaggle v1 completed 36 additional fits while retaining the original
classifier and coarse quantile models. Relative MAE gains range from -0.0301%
to +0.0248%. Both 1h cases regress; 6h changes sign across folds; 24h gains are
negligible. WAPE remains 40.46-46.97%. No validation refit or production change.

All 48 artifact files passed integrity checks. Full-population metrics and axes
were reconstructed; 72 history-only original/refined predictions match cloud
output exactly. Paired spatial intervals for every gain cross zero. They are
conditional on reused windows and do not prove future-time generalization.

Review `reports/hurdle_quantile_resolution_analysis.md`, the comparison CSV,
cluster intervals and verification JSON. Preserve this negative result rather
than selecting individual favorable horizons. This summary does not train;
the original hosted execution snapshot is unchanged.
